In [3]:
import os
import numpy as np
import nibabel as nib
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---------------- CONFIG ----------------
IMG_SIZE = 224
BASE_PATH = "/kaggle/input/datasets/rksrank1/pancreatic-cancer/Task07_Pancreas"
IMAGE_DIR = os.path.join(BASE_PATH, "imagesTr")
MASK_DIR = os.path.join(BASE_PATH, "labelsTr")

# ---------------- FILE LOADING ----------------
all_patients = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if (f.endswith(".nii") or f.endswith(".nii.gz")) and not f.startswith(".")
])

train_pats, test_pats = train_test_split(all_patients, test_size=0.2, random_state=42)

# ---------------- BALANCED SLICE GENERATOR ----------------
def balanced_slice_generator(patient_list, is_training=True):
    """
    Solves Imbalance: Only yields healthy slices occasionally, 
    but always yields tumor slices.
    """
    while True: # Loop indefinitely for the .repeat() function
        for pat in patient_list:
            try:
                img_vol = nib.load(os.path.join(IMAGE_DIR, pat)).get_fdata()
                mask_vol = nib.load(os.path.join(MASK_DIR, pat)).get_fdata()
                
                for i in range(img_vol.shape[2]):
                    mask_slice = mask_vol[:, :, i]
                    is_tumor_slice = np.any(mask_slice == 2) # Label 2 is Pancreas Tumor
                    
                    # BALANCING LOGIC:
                    # If it's a tumor, we take it. 
                    # If it's healthy, we only take it 10% of the time (Downsampling)
                    if is_tumor_slice or (not is_tumor_slice and np.random.random() < 0.1):
                        slice_img = img_vol[:, :, i]
                        slice_img = np.clip(slice_img, -100, 200) # Soft tissue windowing
                        slice_img = (slice_img + 100) / 300.0
                        slice_img = np.stack([slice_img]*3, axis=-1)
                        slice_img = tf.image.resize(slice_img, (IMG_SIZE, IMG_SIZE))
                        
                        yield slice_img, int(is_tumor_slice)
            except Exception:
                continue

# ---------------- DATASETS WITH DEFINED LENGTH ----------------
# We estimate steps_per_epoch to give Keras a "finish line"
BATCH_SIZE = 8
STEPS_PER_EPOCH = 500 

train_ds = tf.data.Dataset.from_generator(
    lambda: balanced_slice_generator(train_pats, is_training=True),
    output_signature=(
        tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_generator(
    lambda: balanced_slice_generator(test_pats, is_training=False),
    output_signature=(
        tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ---------------- MODEL ----------------
def build_model():
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = True
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = Model(inputs=base.input, outputs=out)
    
    # Use standard BinaryCrossentropy now that data is balanced
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), 
                  loss='binary_crossentropy', 
                  metrics=['accuracy', tf.keras.metrics.Recall()])
    return model

model = build_model()

# ---------------- TRAINING WITH STEPS ----------------
model.fit(
    train_ds, 
    validation_data=test_ds, 
    epochs=5, 
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=100
)

# ---------------- PATIENT-LEVEL EVALUATION ----------------
# (Evaluates the WHOLE patient, even the small tumors)
def evaluate_patient_level(patient_list):
    y_true, y_pred = [], []
    for pat in patient_list:
        mask_vol = nib.load(os.path.join(MASK_DIR, pat)).get_fdata()
        y_true.append(1 if np.any(mask_vol == 2) else 0)
        
        # We scan the patient to see if the model detects ANY tumor slice
        img_vol = nib.load(os.path.join(IMAGE_DIR, pat)).get_fdata()
        slice_scores = []
        for i in range(0, img_vol.shape[2], 2):
            img = np.clip(img_vol[:,:,i], -100, 200)
            img = (img + 100) / 300.0
            img = np.stack([img]*3, axis=-1)
            img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
            score = model.predict(tf.expand_dims(img, 0), verbose=0)[0][0]
            slice_scores.append(score)
        
        # If the highest score is > 0.5, the patient has cancer
        y_pred.append(1 if np.max(slice_scores) > 0.5 else 0)
    return y_true, y_pred

y_t, y_p = evaluate_patient_level(test_pats)
print(classification_report(y_t, y_p))

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 948s 2s/step - accuracy: 0.5473 - loss: 0.6895 - recall: 0.6798 - val_accuracy: 0.5450 - val_loss: 0.6858 - val_recall: 0.3499
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 530s 1s/step - accuracy: 0.6898 - loss: 0.6079 - recall: 0.8128 - val_accuracy: 0.5500 - val_loss: 0.6604 - val_recall: 0.2300
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 466s 934ms/step - accuracy: 0.7147 - loss: 0.5696 - recall: 0.8075 - val_accuracy: 0.5000 - val_loss: 0.6944 - val_recall: 0.0221
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 413s 828ms/step - accuracy: 0.7616 - loss: 0.5259 - recall: 0.8440 - val_accuracy: 0.4988 - val_loss: 0.6875 - val_recall: 0.0124
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 390s 782ms/step - accuracy: 0.7903 - loss: 0.4946 - recall: 0.8628 - val_accuracy: 0.4963 - val_loss: 0.8185 - val_recall: 0.0000e+00
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       0.0
           1       0.00      0.00      0.00     

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [1]:
import os
import numpy as np
import cv2
import nibabel as nib
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# --- CONFIGURATION ---
IMG_SIZE = 224
BATCH_SIZE = 16
# Ensure these paths match your Kaggle environment
BASE_PATH = "/kaggle/input/datasets/rksrank1/pancreatic-cancer/Task07_Pancreas"
IMAGE_DIR = os.path.join(BASE_PATH, "imagesTr")
MASK_DIR = os.path.join(BASE_PATH, "labelsTr")

# 1. FILE LOADING (Cleaning hidden files)
all_files = sorted([f for f in os.listdir(IMAGE_DIR) 
                    if (f.endswith(".nii") or f.endswith(".nii.gz")) 
                    and not f.startswith(".")])

train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=42)

# --- BALANCED DATA SEQUENCE ---
class PancreasDataSequence(Sequence):
    def __init__(self, patient_files, batch_size=BATCH_SIZE, is_training=True):
        self.patient_files = patient_files
        self.batch_size = batch_size
        self.is_training = is_training
        self.samples = self._prepare_balanced_index()

    def _prepare_balanced_index(self):
        tumor_slices = []
        healthy_slices = []
        
        print(f"Indexing slices for {'Training' if self.is_training else 'Testing'}...")
        for file in self.patient_files:
            mask_path = os.path.join(MASK_DIR, file)
            mask_vol = nib.load(mask_path).get_fdata()
            for i in range(mask_vol.shape[2]):
                # Label 2 is the tumor in Task07
                if np.any(mask_vol[:, :, i] == 2):
                    tumor_slices.append((file, i, 1))
                else:
                    healthy_slices.append((file, i, 0))
        
        if self.is_training:
            # Fix Heavy Imbalance: Under-sample healthy to match tumor count
            np.random.shuffle(healthy_slices)
            healthy_slices = healthy_slices[:len(tumor_slices)]
            print(f"Balanced Dataset: {len(tumor_slices)} Tumor / {len(healthy_slices)} Healthy")
        
        all_samples = tumor_slices + healthy_slices
        np.random.shuffle(all_samples)
        return all_samples

    def __len__(self):
        return int(np.ceil(len(self.samples) / self.batch_size))

    def __getitem__(self, idx):
        batch = self.samples[idx * self.batch_size : (idx + 1) * self.batch_size]
        X, y = [], []
        
        for file, slice_idx, label in batch:
            # Load image volume
            img_path = os.path.join(IMAGE_DIR, file)
            img_vol = nib.load(img_path).get_fdata()
            slice_img = img_vol[:, :, slice_idx]
            
            # --- METHODOLOGY: PREPROCESSING ---
            # 1. Hounsfield Windowing (-100 to 200) for soft tissue contrast
            slice_img = np.clip(slice_img, -100, 200)
            # 2. Normalization (0 to 1)
            slice_img = (slice_img + 100) / 300.0
            # 3. Resize and convert to 3-channel for EfficientNet
            slice_img = cv2.resize(slice_img, (IMG_SIZE, IMG_SIZE))
            slice_img = np.stack([slice_img]*3, axis=-1)
            
            X.append(slice_img)
            y.append(label)
            
        return np.array(X), np.array(y)

# --- 2. BUILD EFFICIENTNET-B0 ---
def build_efficientnet():
    # Load pre-trained weights
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    
    # Allow fine-tuning to detect subtle <2cm tumor textures
    base_model.trainable = True 
    
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x) # Prevent overfitting on small dataset
    x = Dense(128, activation='relu')(x)
    output = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=base_model.input, outputs=output)
    
    # Use a small learning rate for medical fine-tuning
    model.compile(optimizer=Adam(learning_rate=1e-5), 
                  loss='binary_crossentropy', 
                  metrics=['accuracy', tf.keras.metrics.Recall(name='recall')])
    return model

# --- 3. EXECUTION ---
train_seq = PancreasDataSequence(train_files, is_training=True)
test_seq = PancreasDataSequence(test_files, is_training=False)

model = build_efficientnet()

print("\nStarting EfficientNet-B0 Training...")
model.fit(train_seq, validation_data=test_seq, epochs=10)

# --- 4. FINAL EVALUATION ---
print("\nGenerating Classification Report...")
y_true, y_pred = [], []
for i in range(len(test_seq)):
    X_batch, y_batch = test_seq[i]
    preds = model.predict(X_batch, verbose=0)
    y_true.extend(y_batch)
    y_pred.extend((preds > 0.5).astype(int).flatten())

print(classification_report(y_true, y_pred))

2026-03-25 06:26:34.453908: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774419994.691914      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774419994.749048      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774419995.244423      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774419995.244475      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774419995.244478      55 computation_placer.cc:177] computation placer alr

Indexing slices for Training...
Balanced Dataset: 2067 Tumor / 2067 Healthy
Indexing slices for Testing...


I0000 00:00:1774420098.753953      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Starting EfficientNet-B0 Training...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1774420239.260992     124 service.cc:152] XLA service 0x7c0120121ad0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774420239.261035     124 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1774420246.458769     124 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-03-25 06:30:59.270711: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-25 06:30:59.456508: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-25 06:30:59.917635: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accur

254/259 ━━━━━━━━━━━━━━━━━━━━ 13s 3s/step - accuracy: 0.6434 - loss: 0.6419 - recall: 0.6676

2026-03-25 06:42:44.326211: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-25 06:42:44.511678: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-25 06:42:44.934473: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-25 06:42:45.139745: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


259/259 ━━━━━━━━━━━━━━━━━━━━ 1298s 5s/step - accuracy: 0.6444 - loss: 0.6410 - recall: 0.6689 - val_accuracy: 0.1146 - val_loss: 0.7568 - val_recall: 0.9638
Epoch 2/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 983s 4s/step - accuracy: 0.7802 - loss: 0.4987 - recall: 0.8492 - val_accuracy: 0.2512 - val_loss: 0.7714 - val_recall: 0.8234
Epoch 3/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 975s 4s/step - accuracy: 0.8160 - loss: 0.4338 - recall: 0.8958 - val_accuracy: 0.6678 - val_loss: 0.6512 - val_recall: 0.6128
Epoch 4/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 963s 4s/step - accuracy: 0.8403 - loss: 0.3846 - recall: 0.8984 - val_accuracy: 0.4164 - val_loss: 0.8539 - val_recall: 1.0000
Epoch 5/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 954s 4s/step - accuracy: 0.8601 - loss: 0.3452 - recall: 0.9166 - val_accuracy: 0.4425 - val_loss: 0.7277 - val_recall: 0.9426
Epoch 6/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 1014s 4s/step - accuracy: 0.8721 - loss: 0.3253 - recall: 0.9294 - val_accuracy: 0.4545 - val_loss: 0.8027 - val_recall: 0.9787
Epoch 7